BIBLIOTECAS

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import  f1_score, recall_score, precision_score, accuracy_score
from sklearn.metrics import classification_report

LEITURA DOS ARQUIVOS DESCRITORES DE ETNIAS E LÍNGUAS - CLASSIFICAÇÕES PRÉVIAS E ATUAIS

In [ ]:
caminho_arquivo = 'BDD_etnias_20250505.xlsx';
df_linguas_coleta = pd.read_excel(caminho_arquivo);

caminho_arquivo = 'BDD_linguas_20240604.xlsx';
df_linguas_coleta = pd.read_excel(caminho_arquivo);

caminho_arquivo = 'descritores.xlsx';
df_linguas_novacod = pd.read_excel(caminho_arquivo, sheet_name='etnias');
df_linguas_novacod = pd.read_excel(caminho_arquivo, sheet_name='linguas');

df_ind = pd.read_csv('df_ind_distancia_lingua.csv', sep=';', encoding='utf-8-sig')

DEFINIÇÃO DE VARIÁVEIS NUMÉRICAS E CATEGÓRICAS

In [ ]:
variaveis_numericas = ['VAL_LATITUDE','VAL_LONGITUDE','cod_setor','distancia']
variaveis_categoricas= ['txt_lingua_entrada_coleta','desc_cod_etnia_1_novacod','tipo_setor', 'CD_TI', 'sobrenome','descricao_mais_proxima']

DICIONÁRIO

In [ ]:
dicionario_variaveis = {
    'VAL_LATITUDE': 'Latitude',
    'VAL_LONGITUDE': 'Longitude',
    'cod_setor': 'Setor censitário',
    'distancia': 'Distância de Damerau-Levenshtein entrada-descritor',
    'txt_lingua_entrada_coleta': 'Texto de entrada da língua',
    'txt_etnia_entrada_coleta': 'Texto de entrada da etnia',
    'desc_cod_etnia_1_novacod': 'Etnia após codificação',
    'desc_cod_lingua_1_novacod': 'Língua após codificação',
    'tipo_setor': 'Código do tipo do setor',
    'CD_TI': 'Código da terra indígena',
    'sobrenome': 'Sobrenome',
    'descricao_mais_proxima': 'Descritor mais similar por Damerau-Levenshtein',
    'Predicao': 'Língua predição',
    'Real': 'Língua após codificação',
    'Confianca_Predita': 'Confiança'
}


def formata_colunas(tabela):
    tabela = tabela.rename(columns=dicionario_variaveis)
    return tabela

def formata_indexs(tabela):
    tabela = tabela.rename(index=dicionario_variaveis)
    return tabela

dicionario_proc = {
    'num__VAL_LATITUDE': 'Latitude',
    'num__VAL_LONGITUDE': 'Longitude',
    'num__cod_setor': 'Setor censitário',
    'num__distancia': 'Distância de Damerau-Levenshtein entrada-descritor',
    'cat__txt_lingua_entrada_coleta': 'Texto de entrada da língua',
    'cat__txt_etnia_entrada_coleta': 'Texto de entrada da etnia',
    'cat__desc_cod_etnia_1_novacod': 'Etnia após codificação',
    'cat__desc_cod_lingua_1_novacod': 'Língua após codificação',
    'cat__tipo_setor': 'Código do tipo do setor',
    'cat__CD_TI': 'Código da terra indígena',
    'cat__sobrenome': 'Sobrenome',
    'cat__descricao_mais_proxima': 'Descritor mais similar por Damerau-Levenshtein',
    'Predicao': 'Língua predição',
    'Real': 'Língua após codificação',
    'Confianca_Predita': 'Confiança'
}


def formata_proc(tabela, coluna):
    tabela[coluna] = tabela[coluna].replace(dicionario_proc)
    return tabela

def formata_array(vetor):
    substituir = np.vectorize(lambda x: dicionario_proc.get(x, x))
    return substituir(vetor)

HISTOGRAMA DAS VARIÁVEIS NUMÉRICAS

In [ ]:
formata_colunas(df_ind[variaveis_numericas]).hist(bins=50, figsize=(10,6));

CORRELAÇÃO ENTRE VARIÁVEIS CATEGÓRICAS E RÓTULO

In [ ]:
def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    return np.sqrt(phi2corr/min((kcorr-1),(rcorr-1)))

#variaveis_correlacao = ['txt_lingua_entrada_coleta','desc_cod_lingua_1_novacod', 'tipo_setor', 'CD_TI','sobrenome', 'desc_cod_lingua_1_novacod']
variaveis_correlacao = ['txt_lingua_entrada_coleta','desc_cod_etnia_1_novacod', 'tipo_setor', 'CD_TI','descricao_mais_proxima','desc_cod_lingua_1_novacod']

# Matriz de correlação Cramér's V
n_vars = len(variaveis_correlacao)
corr_matrix = pd.DataFrame(np.zeros((n_vars, n_vars)), 
                          index=variaveis_correlacao, 
                          columns=variaveis_correlacao)

for i in range(n_vars):
    for j in range(n_vars):
        corr_matrix.iloc[i, j] = cramers_v(df_ind[variaveis_correlacao[i]], 
                                         df_ind[variaveis_correlacao[j]])

corr_matrix = formata_colunas(corr_matrix)
corr_matrix = formata_indexs(corr_matrix)

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix.round(3), annot=True, cmap='Blues', center=0.5)
plt.title('Correlação entre Variáveis Categóricas (Cramér\'s V)')
plt.tight_layout()
plt.show()

GARANTIR QUE O DATASET DE TREINO TENHA AO MENOS UM REGISTRO DE CADA CLASSIFICAÇÃO RÓTULO

In [ ]:
# Índices com cardinalidade 1
cardinalidade_lingua = df_ind['desc_cod_lingua_1_novacod'].value_counts()
cardinalidade1_lingua = cardinalidade_lingua[cardinalidade_lingua == 1].index

# Registros com cardinalidade 1
df_cardinalidade1 = df_ind[df_ind['desc_cod_lingua_1_novacod'].isin(cardinalidade1_lingua)].copy()

# Registros com cardinalidade diferente de 1
df_cardinalidadedif1 = df_ind[~df_ind['desc_cod_lingua_1_novacod'].isin(cardinalidade1_lingua)].copy()

# Split treino e validacao_teste
X_treino, X_validacao_teste, y_treino, y_validacao_teste = train_test_split(
    df_cardinalidadedif1.drop(columns=['desc_cod_lingua_1_novacod']),  
    df_cardinalidadedif1['desc_cod_lingua_1_novacod'],
    test_size=0.4,
    stratify=df_cardinalidadedif1['desc_cod_lingua_1_novacod']
)

# Inclusão dos registros de cardinalidade 1 no dataset de treino
X_treino = pd.concat([X_treino, df_cardinalidade1.drop(columns=['desc_cod_lingua_1_novacod'])])
y_treino = pd.concat([y_treino, df_cardinalidade1['desc_cod_lingua_1_novacod']])

# Split validacao e teste
X_teste, X_validacao, y_teste, y_validacao = train_test_split(
    X_validacao_teste,  
    y_validacao_teste,
    test_size=0.5,
)

PIPELINES DE PRÉ-PROCESSAMENTO COM ORDINAL E ONE HOT ENCONDERS

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_pipeline_ordinal = Pipeline(steps=[
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

categorical_pipeline_onehot = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_ordinal = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, variaveis_numericas),
        ('cat', categorical_pipeline_ordinal, variaveis_categoricas)
    ]
)

preprocessor_onehot = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, variaveis_numericas),
        ('cat', categorical_pipeline_onehot, variaveis_categoricas)
    ]
)

display(preprocessor_ordinal)
display(preprocessor_onehot)

PIPELINE PARA O MODELO k-NEAREST NEIGHBOURS

In [ ]:
knn = Pipeline(steps=[
    ('preprocessor', preprocessor_onehot),
    ('classifier', KNeighborsClassifier(n_neighbors=6))
])

knn

TREINAMENTO DO MODELO k-NEAREST NEIGHBOURS

In [ ]:
knn.fit(X_treino, y_treino)

GRAVAÇÃO DO MODELO K-NEAREST NEIGHBOURS

In [ ]:
joblib.dump(knn, 'knn_lingua.pkl')

PREDIÇÕES DE TREINO k-NEAREST NEIGHBOURS

In [ ]:
y_pred_treino = knn.predict(X_treino)
y_pred_treino

MÉTRICAS DE TREINO k-NEAREST NEIGHBOURS

In [ ]:
print(f"Acurácia: {accuracy_score(y_treino, y_pred_treino):.2%}")
print(f"Precisão: {precision_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(f"Revocação: {recall_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(f"F1 Score: {f1_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(classification_report(y_treino, y_pred_treino, zero_division=0))

PREDIÇÕES DE VALIDAÇÃO k-NEAREST NEIGHBOURS

In [ ]:
y_pred_validacao = knn.predict(X_validacao)
y_pred_validacao

MÉTRICAS DE VALIDAÇÃO k-NEAREST NEIGHBOURS

In [ ]:
print(f"Acurácia: {accuracy_score(y_validacao, y_pred_validacao):.2%}")
print(f"Precisão: {precision_score(y_validacao, y_pred_validacao, average='macro', zero_division=0):.2%}")
print(f"Revocação: {recall_score(y_validacao, y_pred_validacao, average='macro', zero_division=0):.2%}")
print(f"F1 Score: {f1_score(y_validacao, y_pred_validacao, average='macro', zero_division=0):.2%}")
print(classification_report(y_validacao, y_pred_validacao, zero_division=0))

PIPELINE PARA O MODELO ÁRVORE DE DECISÃO

In [ ]:
arvoredecisao = Pipeline(steps=[
    ('preprocessor', preprocessor_ordinal),
    ('classifier', DecisionTreeClassifier())
])

arvoredecisao

TREINAMENTO DO MODELO ÁRVORE DE DECISÃO

In [ ]:
arvoredecisao.fit(X_treino, y_treino)

GRAVAÇÃO DO MODELO ÁRVORE DE DECISÃO

In [ ]:
joblib.dump(arvoredecisao, 'arvoredecisao_lingua.pkl')

PREDIÇÕES DE TREINO ÁRVORE DE DECISÃO

In [ ]:
y_pred_treino = arvoredecisao.predict(X_treino)
y_pred_treino

MÉTRICAS DE TREINO ÁRVORE DE DECISÃO

In [ ]:
print(f"Acurácia: {accuracy_score(y_treino, y_pred_treino):.2%}")
print(f"Precisão: {precision_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(f"Revocação: {recall_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(f"F1 Score: {f1_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(classification_report(y_treino, y_pred_treino, zero_division=0))

PREDIÇÕES DE VALIDAÇÃO ÁRVORE DE DECISÃO

In [ ]:
y_pred_validacao = arvoredecisao.predict(X_validacao)
y_pred_validacao

MÉTRICAS DE VALIDAÇÃO ÁRVORE DE DECISÃO

In [ ]:
print(f"Acurácia: {accuracy_score(y_validacao, y_pred_validacao):.2%}")
print(f"Precisão: {precision_score(y_validacao, y_pred_validacao, average='macro', zero_division=0):.2%}")
print(f"Revocação: {recall_score(y_validacao, y_pred_validacao, average='macro', zero_division=0):.2%}")
print(f"F1 Score: {f1_score(y_validacao, y_pred_validacao, average='macro', zero_division=0):.2%}")
print(classification_report(y_validacao, y_pred_validacao, zero_division=0))

PIPELINE DO MODELO FLORESTA ALEATÓRIA

In [ ]:
floresta = Pipeline(steps=[
    ('preprocessor', preprocessor_ordinal),
    ('classifier', RandomForestClassifier())
])

floresta

TREINAMENTO DO MODELO FLORESTA ALEATÓRIA

In [ ]:
floresta.fit(X_treino, y_treino)

GRAVAÇÃO DO MODELO FLORESTA ALEATÓRIA

In [ ]:
joblib.dump(floresta, 'floresta_lingua.pkl')

PREDIÇÕES DE TREINO FLORESTA ALEATÓRIA

In [ ]:
y_pred_treino = floresta.predict(X_treino)
y_pred_treino

MÉTRICAS DE TREINO FLORESTA ALEATÓRIA

In [ ]:
print(f"Acurácia: {accuracy_score(y_treino, y_pred_treino):.2%}")
print(f"Precisão: {precision_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(f"Revocação: {recall_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(f"F1 Score: {f1_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(classification_report(y_treino, y_pred_treino, zero_division=0))

PREDIÇÕES DE VALIDAÇÃO FLORESTA ALEATÓRIA

In [ ]:
y_pred_validacao = floresta.predict(X_validacao)
y_pred_validacao

MÉTRICAS DE VALIDAÇÃO FLORESTA ALEATÓRIA

In [ ]:
print(f"Acurácia: {accuracy_score(y_validacao, y_pred_validacao):.2%}")
print(f"Precisão: {precision_score(y_validacao, y_pred_validacao, average='macro', zero_division=0):.2%}")
print(f"Revocação: {recall_score(y_validacao, y_pred_validacao, average='macro', zero_division=0):.2%}")
print(f"F1 Score: {f1_score(y_validacao, y_pred_validacao, average='macro', zero_division=0):.2%}")
print(classification_report(y_validacao, y_pred_validacao, zero_division=0))

PIPELINE PARA SUPPORT VECTOR CLASSIFIER

In [ ]:
svc = Pipeline(steps=[
    ('preprocessor', preprocessor_onehot),
    ('classifier', SVC())
])

svc

TREINAMENTO DO MODELO SUPPORT VECTOR CLASSIFIER

In [ ]:
svc.fit(X_treino, y_treino)

GRAVAÇÃO DO MODELO SUPPORT VECTOR CLASSIFIER

In [ ]:
joblib.dump(svc, 'svc_lingua.pkl')

PREDIÇÕES DE TREINO SUPPORT VECTOR CLASSIFIER

In [ ]:
y_pred_treino = svc.predict(X_treino)
y_pred_treino

MÉTRICAS DE TREINO SUPPORT VECTOR CLASSIFIER

In [ ]:
print(f"Acurácia: {accuracy_score(y_treino, y_pred_treino):.2%}")
print(f"Precisão: {precision_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(f"Revocação: {recall_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(f"F1 Score: {f1_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(classification_report(y_treino, y_pred_treino, zero_division=0))

PREDIÇÕES DE VALIDAÇÃO SUPPORT VECTOR CLASSIFIER

In [ ]:
y_pred_validacao = svc.predict(X_validacao)
y_pred_validacao

MÉTRICAS DE VALIDAÇÃO SUPPORT VECTOR CLASSIFIER

In [ ]:
print(f"Acurácia: {accuracy_score(y_validacao, y_pred_validacao):.2%}")
print(f"Precisão: {precision_score(y_validacao, y_pred_validacao, average='macro', zero_division=0):.2%}")
print(f"Revocação: {recall_score(y_validacao, y_pred_validacao, average='macro', zero_division=0):.2%}")
print(f"F1 Score: {f1_score(y_validacao, y_pred_validacao, average='macro', zero_division=0):.2%}")
print(classification_report(y_validacao, y_pred_validacao, zero_division=0))